# Topic Deep Dive

## Business question

Which discovered complaint topics concentrate risk and where should teams intervene first?

In [ ]:
from pathlib import Path

import pandas as pd

scores_path = Path("reports/sample/complaint_scores.csv")
terms_path = Path("reports/sample/topic_terms.csv")
if not scores_path.exists() or not terms_path.exists():
    raise FileNotFoundError(
        "Run `cx-complaints analyze --output-dir reports/sample` first."
    )

scores = pd.read_csv(scores_path)
topic_terms = pd.read_csv(terms_path)

In [ ]:
topic_summary = (
    scores.groupby("topic_id", dropna=False)
    .agg(
        complaints=("complaint_id", "size"),
        avg_risk=("risk_score", "mean"),
        high_risk_share=("risk_level", lambda s: (s == "high").mean() * 100),
    )
    .reset_index()
    .sort_values(["complaints", "avg_risk"], ascending=[False, False])
)
topic_summary["avg_risk"] = topic_summary["avg_risk"].round(2)
topic_summary["high_risk_share"] = topic_summary["high_risk_share"].round(1)
topic_summary.head(12)

In [ ]:
labels = (
    topic_terms.sort_values(["topic_id", "rank"])
    .groupby("topic_id")["term"]
    .apply(lambda terms: ", ".join(terms.head(6).astype(str)))
    .rename("topic_label")
    .reset_index()
)
topic_cards = topic_summary.merge(labels, on="topic_id", how="left")
topic_cards.head(12)

In [ ]:
top_topic_ids = topic_cards.head(3)["topic_id"].dropna().astype(int).tolist()
samples = scores[scores["topic_id"].isin(top_topic_ids)].copy()
samples = samples.sort_values(["topic_id", "risk_score"], ascending=[True, False])
samples[
    [
        "topic_id",
        "risk_score",
        "product",
        "issue",
        "company",
        "complaint_what_happened",
    ]
].groupby("topic_id").head(3)

## Topic actions

After running the cells, document:

1. Top 2-3 topics by complaint volume and risk share.
2. Product/company concentration inside those topics.
3. Evidence-backed remediation actions and owners.